# 03. KuaiRec Head-MLP Pretraining for KuaiRand Ranking

**Goal.** Train a behavior-head MLP on KuaiRec, but restrict the input feature space to the features that can also be constructed in KuaiRand. This creates the old behavior-score recommender in the KuaiRand feature space.

The trained ranker has two parts:

1. This notebook trains the head model
   `features -> (p_complete, p_long, p_rewatch, p_neg)`.
2. A later scoring notebook can combine the predicted heads with the already-estimated score weights from `head_weight_pl_weights.json`.

The model is trained on **KuaiRec**, not KuaiRand. KuaiRand random rows are used only to define the transferable feature contract.


## Data, Labels, and Feature Contract

**Training table.** The supervised rows are the KuaiRec processed interaction rows in:

```text
KuaiRec 2.0/data/processed/gnn_data.pt
```

That tensor package was built from:

```text
KuaiRec 2.0/data/processed/processed_data.parquet
```

**KuaiRand feature contract.** The feature list is read from:

```text
python_code_KuaiRand/outputs/feature_build/kuairand_random_prediction_feature_manifest.json
```

The default training feature set is `prediction_feature_cols`, not every saved diagnostic column. This default gives:

| Feature block | Count |
|---|---:|
| Continuous | 22 |
| Binary | 6 |
| Categorical | 33 |
| Total | 61 |

**Labels.** The four labels are observed KuaiRec engagement heads:

| Label | Definition used in the KuaiRec pipeline |
|---|---|
| `y_complete` | `1{watch_ratio >= 1}` |
| `y_long` | `1{play_duration >= 12000 ms or watch_ratio >= 1.5}` |
| `y_rewatch` | `1{watch_ratio >= 2}` |
| `y_neg` | `1{play_duration <= 2000 ms}` |

**Dropped by design.** `i_video_tag_id` and `i_video_tag_name` are not used because KuaiRand raw tags cannot be reliably matched to KuaiRec tag ids/names. The full Modal output also contains `hist_author_recency_days`, but the manifest marks it as model-only and drops it from `prediction_feature_cols`; this notebook follows that default. The dropped model-only feature list is: `['hist_author_recency_days']`.


## Full MLP Feature Table

The table below is the complete default feature set used by this notebook. These are the columns that must be present when scoring KuaiRand random rows later.

| Group | Feature | How it enters the MLP |
|---|---|---|
| Continuous | `u_follow_user_num` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_fans_user_num` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_friend_user_num` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_register_days` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_follow_user_num_log1p` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_fans_user_num_log1p` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_friend_user_num_log1p` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `u_register_days_log1p` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `i_aspect_ratio` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `i_video_duration` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `i_age_since_upload_days` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `ctx_hour_sin` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `ctx_hour_cos` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_ema_y_complete` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_ema_y_long` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_ema_y_rewatch` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_ema_y_neg` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_ema_watchratio` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_cat_ema_complete` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_cat_entropy_l2` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_prev_sess_len` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Continuous | `hist_intersess_gap_h` | Standardized numeric scalar using KuaiRec train-split mean/std. |
| Binary | `u_is_lowactive_period` | Cast to a 0/1 float indicator. |
| Binary | `u_is_live_streamer` | Cast to a 0/1 float indicator. |
| Binary | `u_is_video_author` | Cast to a 0/1 float indicator. |
| Binary | `ctx_is_weekend` | Cast to a 0/1 float indicator. |
| Binary | `hist_last_complete_author` | Cast to a 0/1 float indicator. |
| Binary | `hist_has_author_history` | Cast to a 0/1 float indicator. |
| Categorical | `burst_id` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `session` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_user_active_degree` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_follow_user_num_range` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_fans_user_num_range` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_friend_user_num_range` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_register_days_range` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat0` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat1` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat2` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat3` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat4` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat5` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat6` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat7` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat8` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat9` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat10` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat11` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat12` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat13` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat14` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat15` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat16` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `u_onehot_feat17` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_author_id` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_video_type` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_upload_type` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_visible_status` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_music_id` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_cat_level1_id` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_cat_level2_id` | Encoded as an integer id and passed through a learned embedding. |
| Categorical | `i_cat_level3_id` | Encoded as an integer id and passed through a learned embedding. |


## Training Controls

Before running the notebook, choose the early-stopping controls in the setup cell:

| Parameter | Meaning |
|---|---|
| `NUM_EPOCHS` | Maximum number of epochs to allow. Training may stop earlier. |
| `EARLY_STOPPING_PATIENCE` | Number of consecutive epochs without enough validation-loss improvement before stopping. |
| `MIN_VALID_LOSS_DELTA` | Minimum validation BCE decrease required to count as an improvement. |

Example: if `NUM_EPOCHS = 100` and `EARLY_STOPPING_PATIENCE = 8`, the model can train for at most 100 epochs, but it stops once validation BCE has failed to improve by at least `MIN_VALID_LOSS_DELTA` for 8 epochs. Set `EARLY_STOPPING_PATIENCE = None` or `0` to disable early stopping and run the full epoch budget.


In [1]:
from pathlib import Path
import copy
import json
import math
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
KUAIRAND_DIR = PROJECT_ROOT / "python_code_KuaiRand"
KUAIREC_PROCESSED_DIR = PROJECT_ROOT / "KuaiRec 2.0" / "data" / "processed"

USE_TINY = False
GNN_DATA_PATH = KUAIREC_PROCESSED_DIR / ("gnn_data_tiny.pt" if USE_TINY else "gnn_data.pt")
KUAIREC_SOURCE_TABLE_PATH = KUAIREC_PROCESSED_DIR / "processed_data.parquet"
KUAIRAND_FEATURE_MANIFEST_PATH = KUAIRAND_DIR / "outputs" / "feature_build" / "kuairand_random_prediction_feature_manifest.json"
HEAD_WEIGHT_PATH = PROJECT_ROOT / "python_code_new" / "outputs" / "head_weight_pl_weights.json"

OUTPUT_DIR = KUAIRAND_DIR / "model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "kuairec_transfer_head_mlp_kuairand_features"
MODEL_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}.pt"
METRICS_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_metrics.json"
HISTORY_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_history.csv"
FEATURE_SPEC_OUTPUT_PATH = OUTPUT_DIR / f"{RUN_NAME}_feature_spec.json"

RANDOM_SEED = 20260706
BATCH_SIZE = 16384
EVAL_BATCH_SIZE = 131072
# User-controlled early-stopping settings.
# NUM_EPOCHS is the maximum epoch budget; early stopping may stop earlier.
# Set EARLY_STOPPING_PATIENCE to None or 0 to disable early stopping.
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
MIN_VALID_LOSS_DELTA = 1e-5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT = 0.10
HIDDEN_SIZES = [256, 128, 64]
EMBEDDING_DIM_CAP = 32

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

for path in [GNN_DATA_PATH, KUAIRAND_FEATURE_MANIFEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("device:", DEVICE)
print("KuaiRec tensor package:", GNN_DATA_PATH)
print("KuaiRand feature manifest:", KUAIRAND_FEATURE_MANIFEST_PATH)
print("outputs:", OUTPUT_DIR)

device: mps
KuaiRec tensor package: /Users/haozhangao/Desktop/RecSys Research/KuaiRec 2.0/data/processed/gnn_data.pt
KuaiRand feature manifest: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/outputs/feature_build/kuairand_random_prediction_feature_manifest.json
outputs: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs


In [2]:
manifest = json.loads(KUAIRAND_FEATURE_MANIFEST_PATH.read_text())
label_cols = list(manifest.get("label_cols", ["y_complete", "y_long", "y_rewatch", "y_neg"]))
prediction_feature_cols = list(manifest["prediction_feature_cols"])

continuous_cols = [c for c in manifest["continuous_cols"] if c in prediction_feature_cols]
binary_cols = [c for c in manifest["binary_cols"] if c in prediction_feature_cols]
categorical_cols = [c for c in manifest["categorical_cols"] if c in prediction_feature_cols]
feature_cols = continuous_cols + binary_cols + categorical_cols

if feature_cols != prediction_feature_cols:
    raise ValueError("Feature block ordering does not match manifest['prediction_feature_cols']")

forbidden = set(label_cols) | {
    "watch_ratio", "play_duration", "play_time_ms", "duration_ms", "duration_for_ratio_ms",
    "i_video_tag_id", "i_video_tag_name", "EU", "EU_hat", "u_star", "score", "score_vanilla",
}
leaked = sorted(set(feature_cols) & forbidden)
if leaked:
    raise ValueError(f"Forbidden columns are in the feature set: {leaked}")

print("labels:", label_cols)
print("continuous:", len(continuous_cols))
print("binary:", len(binary_cols))
print("categorical:", len(categorical_cols))
print("total features:", len(feature_cols))
print("dropped unsafe features:", manifest.get("dropped_unsafe_features"))
print("model-only dropped features:", sorted(set(manifest["model_feature_cols"]) - set(prediction_feature_cols)))

labels: ['y_complete', 'y_long', 'y_rewatch', 'y_neg']
continuous: 22
binary: 6
categorical: 33
total features: 61
dropped unsafe features: ['i_video_tag_id', 'i_video_tag_name']
model-only dropped features: ['hist_author_recency_days']


In [3]:
data = torch.load(GNN_DATA_PATH, map_location="cpu", weights_only=False)
metadata = data["feature_metadata"]

source_continuous_cols = list(metadata["continuous_cols"])
source_binary_cols = list(metadata["binary_cols"])
source_categorical_cols = list(metadata["categorical_cols"])
source_label_cols = list(data.get("label_names", label_cols))

missing_cont = sorted(set(continuous_cols) - set(source_continuous_cols))
missing_bin = sorted(set(binary_cols) - set(source_binary_cols))
missing_cat = sorted(set(categorical_cols) - set(source_categorical_cols))
missing_labels = sorted(set(label_cols) - set(source_label_cols))
if missing_cont or missing_bin or missing_cat or missing_labels:
    raise KeyError({
        "missing_continuous": missing_cont,
        "missing_binary": missing_bin,
        "missing_categorical": missing_cat,
        "missing_labels": missing_labels,
    })

cont_idx = torch.tensor([source_continuous_cols.index(c) for c in continuous_cols], dtype=torch.long)
bin_idx = torch.tensor([source_binary_cols.index(c) for c in binary_cols], dtype=torch.long)
cat_idx = torch.tensor([source_categorical_cols.index(c) for c in categorical_cols], dtype=torch.long)
label_idx = torch.tensor([source_label_cols.index(c) for c in label_cols], dtype=torch.long)

all_categorical_cardinalities = list(metadata["categorical_cardinalities"])
categorical_cardinalities = [int(all_categorical_cardinalities[source_categorical_cols.index(c)]) for c in categorical_cols]

x_cont_all = data["x_cont"].cpu()
x_bin_all = data["x_bin"].cpu()
x_cat_all = data["x_cat"].cpu()
y_all = data["y"].cpu()
train_idx = data["train_idx"].cpu().long()
val_idx = data["val_idx"].cpu().long()
test_idx = data["test_idx"].cpu().long()

if y_all.shape[1] != len(source_label_cols):
    raise ValueError("Label tensor width does not match label metadata")

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [int(train_idx.numel()), int(val_idx.numel()), int(test_idx.numel())],
})
split_summary["share"] = split_summary["rows"] / split_summary["rows"].sum()

target_means = []
for split_name, idx in [("train", train_idx), ("validation", val_idx), ("test", test_idx)]:
    y_split = y_all.index_select(0, idx).index_select(1, label_idx).float()
    for j, label in enumerate(label_cols):
        target_means.append({"split": split_name, "label": label, "mean": float(y_split[:, j].mean())})

display(split_summary)
display(pd.DataFrame(target_means).pivot(index="label", columns="split", values="mean"))
print("x_cont_all:", tuple(x_cont_all.shape), "selected:", len(cont_idx))
print("x_bin_all:", tuple(x_bin_all.shape), "selected:", len(bin_idx))
print("x_cat_all:", tuple(x_cat_all.shape), "selected:", len(cat_idx))
print("y_all:", tuple(y_all.shape), "selected labels:", label_cols)

,split,rows,share
0,train,12059684,0.904972
1,validation,803934,0.060328
2,test,462413,0.034700


split,test,train,validation
label,,,
y_complete,0.333291,0.338106,0.315319
y_long,0.230558,0.207596,0.213025
y_neg,0.131577,0.114601,0.117323
y_rewatch,0.082848,0.070785,0.070975


x_cont_all: (13326031, 23) selected: 22
x_bin_all: (13326031, 6) selected: 6
x_cat_all: (13326031, 35) selected: 33
y_all: (13326031, 4) selected labels: ['y_complete', 'y_long', 'y_rewatch', 'y_neg']


In [4]:
def make_loader(edge_ids, batch_size, shuffle):
    ds = TensorDataset(edge_ids.cpu().long())
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=False)

train_loader = make_loader(train_idx, BATCH_SIZE, shuffle=True)
val_loader = make_loader(val_idx, EVAL_BATCH_SIZE, shuffle=False)
test_loader = make_loader(test_idx, EVAL_BATCH_SIZE, shuffle=False)


def fetch_batch(edge_ids):
    edge_ids = edge_ids.cpu().long()
    xb_cont = x_cont_all.index_select(0, edge_ids).index_select(1, cont_idx).float()
    xb_bin = x_bin_all.index_select(0, edge_ids).index_select(1, bin_idx).float()
    xb_cat = x_cat_all.index_select(0, edge_ids).index_select(1, cat_idx).long()
    yb = y_all.index_select(0, edge_ids).index_select(1, label_idx).float()
    return xb_cont.to(DEVICE), xb_bin.to(DEVICE), xb_cat.to(DEVICE), yb.to(DEVICE)

sample_batch = next(iter(train_loader))[0]
xb_cont, xb_bin, xb_cat, yb = fetch_batch(sample_batch)
print("sample batch shapes:", tuple(xb_cont.shape), tuple(xb_bin.shape), tuple(xb_cat.shape), tuple(yb.shape))

sample batch shapes: (16384, 22) (16384, 6) (16384, 33) (16384, 4)


## MLP Structure

The model is an edge-level tabular MLP. For each interaction row, it builds three input blocks:

| Block | Encoding |
|---|---|
| Continuous features | Already standardized in the KuaiRec tensor package using train-split statistics. The selected train-split scaler is saved for KuaiRand inference. |
| Binary features | Stored as `0/1` floats. |
| Categorical features | Stored as integer category codes with `0 = unknown`; each column has its own embedding table. |

Categorical embedding dimensions use:

```text
embedding_dim_c = min(32, max(4, round(sqrt(cardinality_c))))
```

The concatenated vector is passed through:

```text
Linear(input_dim, 256) -> SiLU -> Dropout(0.10)
Linear(256, 128)       -> SiLU -> Dropout(0.10)
Linear(128, 64)        -> SiLU -> Dropout(0.10)
Linear(64, 4)
```

The four output logits correspond to `y_complete`, `y_long`, `y_rewatch`, and `y_neg`. Training uses `BCEWithLogitsLoss` averaged over the four heads. Early stopping monitors validation BCE loss with the user-chosen `NUM_EPOCHS`, `EARLY_STOPPING_PATIENCE`, and `MIN_VALID_LOSS_DELTA` controls, and diagnostics report AUC and average precision for each head.


In [5]:
def default_embedding_dim(cardinality, cap=EMBEDDING_DIM_CAP):
    return min(int(cap), max(4, int(round(math.sqrt(int(cardinality))))))


class EdgeFeatureEncoder(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, emb_dim_cap=32):
        super().__init__()
        self.n_cont = int(n_cont)
        self.n_bin = int(n_bin)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]
        self.embedding_dims = [default_embedding_dim(c, emb_dim_cap) for c in self.categorical_cardinalities]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])
        self.output_dim = self.n_cont + self.n_bin + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.n_cont:
            parts.append(x_cont_batch.float())
        if self.n_bin:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=1)


class TransferHeadMLP(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, hidden_sizes, dropout=0.1, out_dim=4):
        super().__init__()
        self.encoder = EdgeFeatureEncoder(n_cont, n_bin, categorical_cardinalities)
        layers = []
        dim = self.encoder.output_dim
        for hidden in hidden_sizes:
            layers.append(nn.Linear(dim, int(hidden)))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(float(dropout)))
            dim = int(hidden)
        layers.append(nn.Linear(dim, int(out_dim)))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        return self.mlp(self.encoder(x_cont_batch, x_bin_batch, x_cat_batch))


model = TransferHeadMLP(
    n_cont=len(continuous_cols),
    n_bin=len(binary_cols),
    categorical_cardinalities=categorical_cardinalities,
    hidden_sizes=HIDDEN_SIZES,
    dropout=DROPOUT,
    out_dim=len(label_cols),
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

print(model)
print("encoder output dim:", model.encoder.output_dim)
print("first 10 embedding dims:", model.encoder.embedding_dims[:10])
print("parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

TransferHeadMLP(
  (encoder): EdgeFeatureEncoder(
    (embeddings): ModuleList(
      (0): Embedding(4, 4)
      (1): Embedding(242, 16)
      (2): Embedding(5, 4)
      (3): Embedding(9, 4)
      (4-6): 3 x Embedding(8, 4)
      (7): Embedding(3, 4)
      (8): Embedding(9, 4)
      (9): Embedding(31, 6)
      (10): Embedding(1077, 32)
      (11): Embedding(14, 4)
      (12): Embedding(11, 4)
      (13): Embedding(4, 4)
      (14): Embedding(48, 7)
      (15): Embedding(341, 18)
      (16): Embedding(8, 4)
      (17): Embedding(6, 4)
      (18-24): 7 x Embedding(4, 4)
      (25): Embedding(7434, 32)
      (26): Embedding(3, 4)
      (27): Embedding(19, 4)
      (28): Embedding(3, 4)
      (29): Embedding(7686, 32)
      (30): Embedding(40, 6)
      (31): Embedding(138, 12)
      (32): Embedding(215, 15)
    )
  )
  (mlp): Sequential(
    (0): Linear(in_features=296, out_features=256, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=256, out_

In [6]:
@torch.no_grad()
def evaluate_model(loader, split_name):
    model.eval()
    total_loss = 0.0
    total_n = 0
    logits_chunks = []
    label_chunks = []
    for (edge_ids,) in loader:
        xb_cont, xb_bin, xb_cat, yb = fetch_batch(edge_ids)
        logits = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(logits, yb)
        n = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * n
        total_n += n
        logits_chunks.append(logits.detach().cpu())
        label_chunks.append(yb.detach().cpu())

    logits_np = torch.cat(logits_chunks, dim=0).numpy()
    y_np = torch.cat(label_chunks, dim=0).numpy()
    prob_np = 1.0 / (1.0 + np.exp(-logits_np))

    auc_by_head = {}
    ap_by_head = {}
    bce_by_head = {}
    eps = 1e-7
    prob_clipped = np.clip(prob_np, eps, 1 - eps)
    for j, label in enumerate(label_cols):
        y_j = y_np[:, j]
        p_j = prob_np[:, j]
        bce_by_head[label] = float(-np.mean(y_j * np.log(prob_clipped[:, j]) + (1 - y_j) * np.log(1 - prob_clipped[:, j])))
        if len(np.unique(y_j)) < 2:
            auc_by_head[label] = None
            ap_by_head[label] = None
        else:
            auc_by_head[label] = float(roc_auc_score(y_j, p_j))
            ap_by_head[label] = float(average_precision_score(y_j, p_j))

    valid_aucs = [v for v in auc_by_head.values() if v is not None]
    valid_aps = [v for v in ap_by_head.values() if v is not None]
    return {
        "split": split_name,
        "loss": total_loss / max(total_n, 1),
        "auc_by_head": auc_by_head,
        "average_precision_by_head": ap_by_head,
        "bce_by_head": bce_by_head,
        "mean_auc": float(np.mean(valid_aucs)) if valid_aucs else None,
        "mean_average_precision": float(np.mean(valid_aps)) if valid_aps else None,
        "rows": total_n,
    }


def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    total_n = 0
    for (edge_ids,) in loader:
        xb_cont, xb_bin, xb_cat, yb = fetch_batch(edge_ids)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb_cont, xb_bin, xb_cat)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        n = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * n
        total_n += n
    return total_loss / max(total_n, 1)

In [7]:
training_history = []
best_state = None
best_valid_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
early_stopped = False
patience_enabled = EARLY_STOPPING_PATIENCE is not None and int(EARLY_STOPPING_PATIENCE) > 0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(train_loader)
    valid_metrics = evaluate_model(val_loader, "validation")

    improved = valid_metrics["loss"] < (best_valid_loss - MIN_VALID_LOSS_DELTA)
    if improved:
        best_valid_loss = float(valid_metrics["loss"])
        best_epoch = int(epoch)
        epochs_without_improvement = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    record = {
        "epoch": int(epoch),
        "train_loss": float(train_loss),
        "valid_loss": float(valid_metrics["loss"]),
        "valid_mean_auc": valid_metrics["mean_auc"],
        "valid_mean_average_precision": valid_metrics["mean_average_precision"],
        "best_valid_loss": float(best_valid_loss),
        "best_epoch": int(best_epoch),
        "improved": bool(improved),
        "epochs_without_improvement": int(epochs_without_improvement),
    }
    for label, value in valid_metrics["auc_by_head"].items():
        record[f"valid_auc_{label}"] = value
    training_history.append(record)
    print(record)

    if patience_enabled and epochs_without_improvement >= int(EARLY_STOPPING_PATIENCE):
        early_stopped = True
        print(
            f"Early stopping at epoch {epoch}; "
            f"best epoch={best_epoch}, best validation BCE={best_valid_loss:.6f}, "
            f"patience={int(EARLY_STOPPING_PATIENCE)}"
        )
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

history_df = pd.DataFrame(training_history)
display(history_df.tail())
print("best epoch:", best_epoch)
print("best validation BCE:", best_valid_loss)

{'epoch': 1, 'train_loss': 0.34968460344946584, 'valid_loss': 0.3246195283077931, 'valid_mean_auc': 0.8422448679722203, 'valid_mean_average_precision': 0.52385313880176, 'best_valid_loss': 0.3246195283077931, 'best_epoch': 1, 'improved': True, 'epochs_without_improvement': 0, 'valid_auc_y_complete': 0.8402300600033188, 'valid_auc_y_long': 0.8188941300538972, 'valid_auc_y_rewatch': 0.8450076003807554, 'valid_auc_y_neg': 0.8648476814509101}
{'epoch': 2, 'train_loss': 0.3248762871556078, 'valid_loss': 0.321127769280994, 'valid_mean_auc': 0.846764873909658, 'valid_mean_average_precision': 0.5318399794043732, 'best_valid_loss': 0.321127769280994, 'best_epoch': 2, 'improved': True, 'epochs_without_improvement': 0, 'valid_auc_y_complete': 0.8458181716822287, 'valid_auc_y_long': 0.8242216756309505, 'valid_auc_y_rewatch': 0.8504812737076598, 'valid_auc_y_neg': 0.866538374617793}
{'epoch': 3, 'train_loss': 0.32136576473990985, 'valid_loss': 0.3181172176102077, 'valid_mean_auc': 0.850408961396354

,epoch,train_loss,valid_loss,valid_mean_auc,valid_mean_average_precision,best_valid_loss,best_epoch,improved,epochs_without_improvement,valid_auc_y_complete,valid_auc_y_long,valid_auc_y_rewatch,valid_auc_y_neg
19,20,0.309726,0.311912,0.858023,0.545882,0.311569,19,False,1,0.858435,0.836862,0.864895,0.871902
20,21,0.309521,0.312834,0.857957,0.544000,0.311569,19,False,2,0.858061,0.837160,0.864566,0.872041
21,22,0.309320,0.311981,0.858310,0.544637,0.311569,19,False,3,0.858695,0.837043,0.865416,0.872086
22,23,0.309151,0.311816,0.858353,0.545712,0.311569,19,False,4,0.859049,0.837171,0.865202,0.871992
23,24,0.308992,0.312059,0.858184,0.545315,0.311569,19,False,5,0.858936,0.837211,0.864424,0.872166


best epoch: 19
best validation BCE: 0.31156918014368445


In [8]:
final_valid_metrics = evaluate_model(val_loader, "validation")
final_test_metrics = evaluate_model(test_loader, "test")

print("validation metrics")
print(json.dumps(final_valid_metrics, indent=2))
print("test metrics")
print(json.dumps(final_test_metrics, indent=2))

validation metrics
{
  "split": "validation",
  "loss": 0.31156918014368445,
  "auc_by_head": {
    "y_complete": 0.8581172782118116,
    "y_long": 0.8376079035083311,
    "y_rewatch": 0.8666293519597751,
    "y_neg": 0.8720317550765686
  },
  "average_precision_by_head": {
    "y_complete": 0.7308547234036695,
    "y_long": 0.5989239098825507,
    "y_rewatch": 0.3769902930439665,
    "y_neg": 0.48600142251306444
  },
  "bce_by_head": {
    "y_complete": 0.4246980845928192,
    "y_long": 0.3797016143798828,
    "y_rewatch": 0.1879696398973465,
    "y_neg": 0.25390732288360596
  },
  "mean_auc": 0.8585965721891217,
  "mean_average_precision": 0.5481925872108128,
  "rows": 803934
}
test metrics
{
  "split": "test",
  "loss": 0.34624212500146817,
  "auc_by_head": {
    "y_complete": 0.8392543430365335,
    "y_long": 0.8110536942287578,
    "y_rewatch": 0.825719298433628,
    "y_neg": 0.8618810193379051
  },
  "average_precision_by_head": {
    "y_complete": 0.7188761089990136,
    "y_long

## Save Transfer-Ranker Artifacts

The saved checkpoint is meant for the next KuaiRand scoring notebook. It contains:

| Artifact | Contents |
|---|---|
| `kuairec_transfer_head_mlp_kuairand_features.pt` | MLP weights, architecture, selected feature metadata, and preprocessing needed to encode KuaiRand rows. |
| `kuairec_transfer_head_mlp_kuairand_features_metrics.json` | Train/validation/test diagnostics, feature lists, label list, and paths. |
| `kuairec_transfer_head_mlp_kuairand_features_history.csv` | Epoch-level training history. |
| `kuairec_transfer_head_mlp_kuairand_features_feature_spec.json` | Human-readable feature contract for the KuaiRand scoring notebook. |

The checkpoint does not consume KuaiRand labels. It only stores the encoder needed to apply the KuaiRec-trained head model to KuaiRand random rows.


In [9]:
def selected_dict(source, keys):
    return {k: source[k] for k in keys if k in source}

selected_feature_metadata = {
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "continuous_scaler": selected_dict(metadata.get("continuous_scaler", {}), continuous_cols),
    "continuous_missing": selected_dict(metadata.get("continuous_missing", {}), continuous_cols),
    "binary_missing": selected_dict(metadata.get("binary_missing", {}), binary_cols),
    "categorical_mappings": selected_dict(metadata.get("categorical_mappings", {}), categorical_cols),
    "categorical_cardinalities": categorical_cardinalities,
    "categorical_encoding": metadata.get("categorical_encoding"),
    "max_cardinality_without_hashing": metadata.get("max_cardinality_without_hashing"),
    "hash_bucket_size": metadata.get("hash_bucket_size"),
    "unknown_code": 0,
}

head_weight_payload = None
if HEAD_WEIGHT_PATH.exists():
    head_weight_payload = json.loads(HEAD_WEIGHT_PATH.read_text())

feature_spec = {
    "run_name": RUN_NAME,
    "trained_on": "KuaiRec",
    "intended_scoring_data": "KuaiRand random recommendations",
    "source_tensor_package": str(GNN_DATA_PATH),
    "source_processed_table": str(KUAIREC_SOURCE_TABLE_PATH),
    "kuairand_manifest_path": str(KUAIRAND_FEATURE_MANIFEST_PATH),
    "feature_contract": "manifest['prediction_feature_cols']",
    "label_cols": label_cols,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
    "feature_cols": feature_cols,
    "dropped_unsafe_features": manifest.get("dropped_unsafe_features", []),
    "model_only_features_not_used": sorted(set(manifest["model_feature_cols"]) - set(prediction_feature_cols)),
    "architecture": {
        "model_class": "TransferHeadMLP",
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
        "embedding_dim_rule": "min(32, max(4, round(sqrt(cardinality))))",
        "embedding_dims": model.encoder.embedding_dims,
        "encoder_output_dim": int(model.encoder.output_dim),
        "output_dim": len(label_cols),
        "loss": "BCEWithLogitsLoss over y_complete, y_long, y_rewatch, y_neg",
    },
    "selected_feature_metadata": selected_feature_metadata,
    "head_weight_path": str(HEAD_WEIGHT_PATH) if HEAD_WEIGHT_PATH.exists() else None,
    "head_weight_payload": head_weight_payload,
}

metrics = {
    "run_name": RUN_NAME,
    "random_seed": RANDOM_SEED,
    "device": DEVICE,
    "use_tiny": bool(USE_TINY),
    "num_epochs_requested": NUM_EPOCHS,
    "num_epochs_run": len(training_history),
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_enabled": bool(patience_enabled),
    "min_valid_loss_delta": MIN_VALID_LOSS_DELTA,
    "early_stopped": bool(early_stopped),
    "best_epoch": int(best_epoch),
    "best_validation_bce_loss": float(best_valid_loss),
    "batch_size": BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "hidden_sizes": HIDDEN_SIZES,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "split_rows": split_summary.to_dict(orient="records"),
    "validation_metrics": final_valid_metrics,
    "test_metrics": final_test_metrics,
    "feature_spec_path": str(FEATURE_SPEC_OUTPUT_PATH),
    "model_output_path": str(MODEL_OUTPUT_PATH),
    "history_output_path": str(HISTORY_OUTPUT_PATH),
    "metrics_output_path": str(METRICS_OUTPUT_PATH),
    "label_cols": label_cols,
    "feature_cols": feature_cols,
    "continuous_cols": continuous_cols,
    "binary_cols": binary_cols,
    "categorical_cols": categorical_cols,
}

history_df.to_csv(HISTORY_OUTPUT_PATH, index=False)
FEATURE_SPEC_OUTPUT_PATH.write_text(json.dumps(feature_spec, indent=2, default=str) + "\n")
METRICS_OUTPUT_PATH.write_text(json.dumps(metrics, indent=2, default=str) + "\n")

torch.save({
    "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "model_class": "TransferHeadMLP",
    "config": feature_spec["architecture"],
    "feature_spec": feature_spec,
    "metrics": metrics,
}, MODEL_OUTPUT_PATH)

print("saved model:", MODEL_OUTPUT_PATH)
print("saved metrics:", METRICS_OUTPUT_PATH)
print("saved history:", HISTORY_OUTPUT_PATH)
print("saved feature spec:", FEATURE_SPEC_OUTPUT_PATH)

saved model: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_head_mlp_kuairand_features.pt
saved metrics: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_head_mlp_kuairand_features_metrics.json
saved history: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_head_mlp_kuairand_features_history.csv
saved feature spec: /Users/haozhangao/Desktop/RecSys Research/python_code_KuaiRand/model_outputs/kuairec_transfer_head_mlp_kuairand_features_feature_spec.json


## Next Step

Use the saved checkpoint to score:

```text
python_code_KuaiRand/outputs/feature_build/kuairand_random_prediction_ranking_input.parquet
```

The scoring notebook should:

1. encode the KuaiRand rows with `feature_spec['selected_feature_metadata']`,
2. predict the four head probabilities,
3. combine them with the PL head weights to form the old behavior score,
4. compare that score ranking against the predicted-EU ranking within the random-exposure groups.
